In [2]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [3]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="skbose/indian-english-nptel-v0", 
    repo_type="dataset", local_dir="./indian-english-nptel-v0", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 269 files: 100%|██████████| 269/269 [01:06<00:00,  4.07it/s]


'/home/ubuntu/indian-english-nptel-v0'

In [4]:
files = glob('indian-english-nptel-v0/*/*.parquet')
len(files)

269

In [5]:
# df = pd.read_parquet(files[0])
# df

,audio,file_name,transcription,speaker_name,transcription_normalised
0,{'bytes': b'RIFF\xc6n\x02\x00WAVEfmt \x10\x00\...,f8e614ba63358b054131e1b25427226a707df6bdcf2062...,STRAIN WAY SO THIS IS DELTA L BY L AND CORRESP...,Dr.S.P.Harsha,strain way so this is delta l by l and corresp...
1,{'bytes': b'RIFF\xc2\x1c\x04\x00WAVEfmt \x10\x...,1f6768e737a05be96645eefa07baa71e2b7a2f63973b73...,ARE YOU KNOW VERY CLEAR SO THIS IS THIS DOES N...,Prof. P.C. Deshmukh,are you know very clear so this is this does n...
2,{'bytes': b'RIFF&\xdd\x04\x00WAVEfmt \x10\x00\...,4fce083e9977d0ea222c7bb16ef055a1abd8533123e507...,WE WILL NOT THINK THAT YOU KNOW THERE IS A GOD...,Prof. P.S. Sastry,we will not think that you know there is a god...
3,{'bytes': b'RIFF&$\x03\x00WAVEfmt \x10\x00\x00...,095a9e2bcb96c9ca197c63d4176a3a7047b2144a0fd68f...,POTENTIAL ENERGY AND AT ANY OTHER THE SYSTEM H...,Prof.S.K.Dwivedy,potential energy and at any other the system h...
4,{'bytes': b'RIFFFk\x03\x00WAVEfmt \x10\x00\x00...,ccaa392ba1afede45b31040345c84e68b60bd2358b1c79...,EXAMPLES CHANNEL FLOW OR EVEN THE GROUND WATER...,Dr. Ashu Jain,examples channel flow or even the ground water...
...,...,...,...,...,...
2020,{'bytes': b'RIFFh\x9c\x05\x00WAVEfmt \x10\x00\...,cb63eb43c1c6e3b6b556ec5dbbc2ffc1f8a50a839c0610...,THIS IS RUNNING AT SIX HUNDRED RPM AND THIS HA...,Prof. A.R. Mohanty,this is running at six hundred rpm and this ha...
2021,{'bytes': b'RIFFF*\x03\x00WAVEfmt \x10\x00\x00...,ef8a5a3e21cb410430150fd6a3868787136729745869c5...,OF CURRENT IDEALLY A INSULATORS SHOULD HAVE A ...,Prof. H.S. Maiti,of current ideally a insulators should have a ...
2022,{'bytes': b'RIFF\x06\xa5\x04\x00WAVEfmt \x10\x...,9e3dfc31bb4df971324087fadb13b6bcf000a34719349b...,A VERY SUCCESSFUL COMMERCIAL PROCESS FOR EXPLO...,Prof.H.S. Ray,a very successful commercial process for explo...
2023,{'bytes': b'RIFF&\x07\x02\x00WAVEfmt \x10\x00\...,dfd81b2cbe7533b2d49d845c713cc0310aa862a2ba8bae...,P LEVEL BECAUSE THE P ORBITALS ARE INVOLVED FOR,Prof. D. Ray,p level because the p orbitals are involved for


In [6]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            t = df['transcription_normalised'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{df['speaker_name'].iloc[i]}"
            })
        
    return data

In [7]:
data = loop((files[:1], 0))

100%|██████████| 1/1 [01:37<00:00, 97.94s/it]


In [ ]:
data = multiprocessing(files, loop, cores = 20)

  0%|          | 0/13 [00:00<?, ?it/s]

In [10]:
len(data)

544167

In [11]:
data[0]

{'audio_filename': 'indian-english-nptel-v0_audio/indian-english-nptel-v0-data-train-00106-of-00215_0.mp3',
 'text': 'strain way so this is delta l by l and corresponding you see we have two times of delta d by d',
 'speaker': 'indian-english-nptel-v0_audio_Dr.S.P.Harsha'}

In [12]:
with open('indian-english-nptel-v0.json', 'w') as fopen:
    json.dump(data, fopen)

In [13]:
audio_files = [d['audio_filename'] for d in data]

with open('indian-english-nptel-v0-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [15]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'indian-english-nptel-v0_audio/indian-english-nptel-v0-data-train-00106-of-00215_0.mp3',
 'text': 'strain way so this is delta l by l and corresponding you see we have two times of delta d by d',
 'speaker': 'indian-english-nptel-v0_audio_Dr.S.P.Harsha'}

In [16]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'indian-english-nptel-v0')

Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00,  8.81ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1): 100%|█████████▉| 35.1MB / 35.2MB,  176MB/s  
Processing Files (1 / 1): 100%|██████████| 35.2MB / 35.2MB, 58.7MB/s  
Processing Files (1 / 1): 100%|██████████| 35.2MB / 35.2MB, 44.0MB/s  
New Data Upload: 100%|██████████| 35.2MB / 35.2MB, 44.0MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.61s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/0c3005a6f7cdac879ece964088a6545ea519c863', commit_message='Upload dataset', commit_description='', oid='0c3005a6f7cdac879ece964088a6545ea519c863', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [19]:
# !zip -rq indian-english-nptel-v0_audio_neucodec.zip indian-english-nptel-v0_audio_neucodec

In [2]:
# !hf upload malaysia-ai/Multilingual-TTS indian-english-nptel-v0_audio_neucodec.zip --repo-type=dataset